# Critical Input DEQN: Fixed Taylor Rule

This notebook trains the fixed-intercept Taylor-rule DEQN network using the frozen natural benchmark checkpoint.

In [ ]:
# Configure paths and fixed Taylor training settings.
from pathlib import Path
import json
import subprocess
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError('No existing checkpoint found: ' + ', '.join(map(str, paths)))

NATURAL_CANDIDATES = [
    ARTIFACT_ROOT / 'natural' / 'checkpoints' / 'natural_best.pt',
    ARTIFACT_ROOT / 'natural' / 'natural.pt',
]
NATURAL_CKPT = next((path for path in NATURAL_CANDIDATES if path.exists()), None)
OUT = ARTIFACT_ROOT / 'fixed_taylor'
OUT.mkdir(parents=True, exist_ok=True)

RULE_STEPS = 8_000
QMC_TRAIN = 256
QMC_VAL = 512
NATURAL_ORACLE_NODES = 32
NATURAL_ORACLE_CHUNK_SIZE = 8192
N_VAL_STATES = 1024
HIDDEN_WIDTH = 192
HIDDEN_DEPTH = 2
LOG_EVERY = 100
BATCH_SIZE = 2048
SIM_BATCH_SIZE = 512
EPISODE_LENGTH = 20
EPISODE_UPDATES_PER_EPISODE = 2
EPISODE_BROAD_SHARE = 0.50
CHECKPOINT_EVERY = 1000
TARGET_RMS = None
TARGET_MAX_ABS = None
TARGET_SCENARIO_Q_RMS = 1e-2
EARLY_STOP_PATIENCE = None
MIN_STEPS_BEFORE_STOP = None
STOP_VAL_STATES = 512
SCENARIO_Q_WEIGHT = 25.0
CALM_ANCHOR_WEIGHT = 5.0
CALM_RESIDUAL_WEIGHT = 5.0
SCENARIO_BURNIN = 5
SCENARIO_HORIZON = 10
SCENARIO_LOSS_INTERVAL = 25
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'

print(ROOT)
print(OUT)

# Stream subprocess output line by line in Colab instead of waiting silently.
def run_stream(cmd, *, cwd=ROOT, env=None):
    print('Running:', ' '.join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)


In [ ]:
# Train the fixed-intercept Taylor rule network.
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.run_train',
    '--output-dir', str(OUT),
    '--policies', 'fixed',
    '--natural-benchmark', 'oracle',
    '--skip-natural-network',
    '--natural-oracle-nodes', str(NATURAL_ORACLE_NODES),
    '--natural-oracle-chunk-size', str(NATURAL_ORACLE_CHUNK_SIZE),
    '--rule-steps', str(RULE_STEPS),
    '--rule-trainer', 'episode',
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--stop-val-states', str(STOP_VAL_STATES),
    '--log-every', str(LOG_EVERY),
    '--batch-size', str(BATCH_SIZE),
    '--sim-batch-size', str(SIM_BATCH_SIZE),
    '--episode-length', str(EPISODE_LENGTH),
    '--episode-updates-per-episode', str(EPISODE_UPDATES_PER_EPISODE),
    '--episode-broad-share', str(EPISODE_BROAD_SHARE),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--rule-scenario-q-weight', str(SCENARIO_Q_WEIGHT),
    '--rule-calm-anchor-weight', str(CALM_ANCHOR_WEIGHT),
    '--rule-calm-residual-weight', str(CALM_RESIDUAL_WEIGHT),
    '--rule-scenario-burnin', str(SCENARIO_BURNIN),
    '--rule-scenario-horizon', str(SCENARIO_HORIZON),
    '--rule-scenario-loss-interval', str(SCENARIO_LOSS_INTERVAL),
    '--target-scenario-q-rms', str(TARGET_SCENARIO_Q_RMS),
]
if TARGET_RMS is not None:
    cmd += ['--target-rms', str(TARGET_RMS)]
if TARGET_MAX_ABS is not None:
    cmd += ['--target-max-abs', str(TARGET_MAX_ABS)]
if EARLY_STOP_PATIENCE is not None:
    cmd += ['--early-stop-patience', str(EARLY_STOP_PATIENCE)]
if MIN_STEPS_BEFORE_STOP is not None:
    cmd += ['--min-steps-before-stop', str(MIN_STEPS_BEFORE_STOP)]
run_stream(cmd, cwd=ROOT)


In [ ]:
# Inspect out-of-sample residual diagnostics for fixed Taylor.
with (OUT / 'fixed_eval.json').open('r', encoding='utf-8') as fh:
    fixed_eval = json.load(fh)
fixed_eval


In [ ]:
# IRF mechanism diagnostics for fixed Taylor.
from src.critical_input_deqn.notebook_diagnostics import rule_ir_mechanism_diagnostics

fixed_ir_labels, fixed_ir_defs, fixed_mechanism_table = rule_ir_mechanism_diagnostics(
    artifact_root=ARTIFACT_ROOT,
    output_dir=OUT,
    policy='fixed',
    natural_checkpoint=NATURAL_CKPT,
    device=DEVICE,
    dtype=DTYPE,
    hidden_width=HIDDEN_WIDTH,
    hidden_depth=HIDDEN_DEPTH,
    natural_benchmark='oracle',
    natural_oracle_nodes=NATURAL_ORACLE_NODES,
    natural_oracle_chunk_size=NATURAL_ORACLE_CHUNK_SIZE,
)
